In [ ]:
import pandas as pd
import os
import numpy as np

In [ ]:
def clean_by_month(df_month):
    df_month['tpep_pickup_datetime'] = pd.to_datetime(df_month['tpep_pickup_datetime'], errors='coerce')
    df_month['tpep_dropoff_datetime'] = pd.to_datetime(df_month['tpep_dropoff_datetime'], errors='coerce')
    # Chuyển các cột số thành chuẩn dạng số
    numeric_cols = ['passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID', 
                'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 
                'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 
                'congestion_surcharge', 'Airport_fee']

    for col in numeric_cols:
        if col in df_month.columns:
            df_month[col] = pd.to_numeric(df_month[col], errors='coerce')
    # Điền khuyết cho các giá giá trị
    if 'store_and_fwd_flag' in df_month.columns:
        df_month['store_and_fwd_flag'] = df_month['store_and_fwd_flag'].fillna('N')
        df_month['store_and_fwd_flag'] = df_month['store_and_fwd_flag'].astype('category')
    
    # Điền giá trị bằng 0 cho các loại phí nếu bị NaN
    cols_to_fill_zero = ['tip_amount', 'tolls_amount', 'Airport_fee', 'congestion_surcharge', 
                   'improvement_surcharge', 'extra', 'mta_tax']
    for col in cols_to_fill_zero:
        if col in df_month.columns:
            df_month[col] = df_month[col].fillna(0)


    # Tạo cờ
    def add_flag(df, condition, flag_name):
        df.loc[condition, 'qa_flags'] = df.loc[condition, 'qa_flags'].apply(lambda x: x + [flag_name])
    df_month['qa_flags'] = [[] for _ in range(len(df_month))]


    # Bắt đầu gắn cờ cho các giá trị bị
    add_flag(df_month,df_month["passenger_count"].isna(), "FLAG_FIXED_MISSINGS_PASSENGER")
    add_flag(df_month,df_month["RatecodeID"].isna(), "FLAG_FIXED_MISSINGS_RatecodeID")
    df_month["passenger_count"] = df_month["passenger_count"].replace(0,1)
    cols_to_fill_mode = 'passenger_count'
    if cols_to_fill_mode in df_month.columns:
        mode_val = df_month[cols_to_fill_mode].mode()[0]
        print(f"Điền NaN cho cột '{cols_to_fill_mode}' bằng giá trị mode: {mode_val}")
        df_month[cols_to_fill_mode] = df_month[cols_to_fill_mode].fillna(mode_val)
    df_month["RatecodeID"] = df_month["RatecodeID"].fillna(99) # Unknow

    # Chuyển về đúng giá trị
    int_cols = ['passenger_count', 'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type']
    for col in int_cols:
        if col in df_month.columns:
            df_month[col] = df_month[col].astype("int32")

    # Check điều kiện loại trả
    payment_type_condition = (df_month["payment_type"] != 1) & (df_month["tip_amount"] > 0)
    add_flag(df_month, payment_type_condition, "FLAG_FIXED_PAYMENT_TYPE")
    if payment_type_condition.sum() > 0:
        print(f"Phát hiện {payment_type_condition.sum()} hàng bị lệch payment type.")
        df_month.loc[payment_type_condition, "tip_amount"] = 0

    # Check điều kiện phí tắc nghẽn
    invalid_congestion_condition = (df_month['congestion_surcharge'] != 0.0) & (df_month['congestion_surcharge'] != 2.5)
    add_flag(df_month, invalid_congestion_condition, "FLAG_INVALID_CONGESTION_IMPROVEMENT")
    if invalid_congestion_condition.sum() > 0:
        print(f"Phát hiện {invalid_congestion_condition.sum()} hàng có congestion surchange không hợp lệ.")

    # Check điều kiện cho thuế mta
    invalid_mta_condition = (df_month['mta_tax'] != 0.0) & (df_month["mta_tax"] != 0.5)
    add_flag(df_month, invalid_mta_condition, "FLAG_INVALID_MTA_TAX")
    if invalid_mta_condition.sum() > 0:
        print(f"Phát hiện {invalid_mta_condition.sum()} hàng có mta_tax không hợp lệ.")

    # Check điều kiện cho phí cải tiến
    invalid_improvement_condition = df_month["improvement_surcharge"] != 1.0
    add_flag(df_month, invalid_improvement_condition, "FLAG_INVALID_IMPROVEMENT_SURCHANGE")
    if invalid_improvement_condition.sum() > 0:
        print(f"Phát hiện {invalid_improvement_condition.sum()} hàng có improvement surchange không hợp lệ.")


    # Check điều kiện cho phí sân bay
    invalid_airport_fee = (df_month["Airport_fee"] != 0.00) & (df_month["Airport_fee"] != 1.25) & (df_month["Airport_fee"] != 1.75)
    add_flag(df_month, invalid_airport_fee, "FLAG_INVALID_AIRPORT_FEE")
    if invalid_airport_fee.sum() > 0:
        print(f"Phát hiện {invalid_airport_fee.sum()} hàng có airport fee không hợp lệ")

    # Check điều kiện cho phí thêm
    invalid_extra_condition = (df_month["extra"] != 0.00) & (df_month["extra"] != 1.00) & (df_month["extra"] != 2.50)
    add_flag(df_month, invalid_extra_condition, "FLAG_INVALID_EXTRA")
    if invalid_extra_condition.sum() > 0:
        print(f"Phát hiện {invalid_extra_condition.sum()} hàng có extra không hợp lệ")


    # Check điều kiện cho tổng phí
    total_condition = abs(df_month["fare_amount"] + df_month["extra"] +  df_month["mta_tax"] + df_month["tip_amount"] + df_month["tolls_amount"] + df_month["improvement_surcharge"] + df_month["congestion_surcharge"] + df_month["Airport_fee"] - df_month["total_amount"]) > 1e-3
    add_flag(df_month, total_condition, "FLAG_FIXED_TOTAL_AMOUNT")
    row_fixed = total_condition.sum()
    if row_fixed > 0:
        print(f"Phát hiện {row_fixed} hàng có total không đúng")
        calculated_sum = (df_month["fare_amount"] + df_month["extra"] +  df_month["mta_tax"] + 
                        df_month["tip_amount"] + df_month["tolls_amount"] + 
                        df_month["improvement_surcharge"] + df_month["congestion_surcharge"] + 
                        df_month["Airport_fee"])
        
        # 2. Ghi đè "total_amount" tại các hàng bị lỗi
        #    bằng giá trị "calculated_sum" tại chính các hàng đó
        df_month.loc[total_condition, "total_amount"] = calculated_sum[total_condition]
        
        print(f"Đã sửa và chuẩn hóa lại {row_fixed} hàng 'total_amount'.")
    
    # Check điều kiện thời gian đón lớn hơn thời gian trả
    swap_condition = df_month["tpep_pickup_datetime"] > df_month["tpep_dropoff_datetime"]
    add_flag(df_month, swap_condition, 'FLAG_FIXED_TIME_SWAPPED')
    rows_swapped = swap_condition.sum()
    if rows_swapped > 0:
        print(f"Phát hiện và gắn cờ {rows_swapped} hàng có thời gian bị ngược.")
        
        cols_to_swap = ['tpep_pickup_datetime', 'tpep_dropoff_datetime']
        swapped_cols = ['tpep_dropoff_datetime', 'tpep_pickup_datetime']
        df_month.loc[swap_condition, cols_to_swap] = df_month.loc[swap_condition, swapped_cols].values
    # xử lý bằng các hoán đổi lại -> do có thể nhầm lẫn trong khâu nhập liệu
    year_condition = (df_month["tpep_pickup_datetime"].dt.year != 2023) | (df_month["tpep_dropoff_datetime"].dt.year != 2023)
    add_flag(df_month, year_condition, "FLAG_YEAR_OUT_OF_RANGE")
    if year_condition.sum() > 0:
        print(f"Phát hiện và gắn cờ {year_condition.sum()} hàng có thời gian ngoài phạm vi.")
    
    # Tính thời gian chuyến đi
    df_month['trip_duration_minutes'] = (df_month['tpep_dropoff_datetime'] - df_month['tpep_pickup_datetime']).dt.total_seconds() / 60
    df_month['speed_mph'] = df_month['trip_distance'] / (df_month['trip_duration_minutes'] / 60 + 1e-6)
    df_month['speed_mph'] = df_month['speed_mph'].replace([np.inf, -np.inf], 0) 

    speed_condition = (df_month["speed_mph"] > 45) # ~ 72km/h
    print(f"Đã phát hiện {speed_condition.sum()} hàng speed quá cao")
    add_flag(df_month, speed_condition, "FLAG_SPEED_TOO_HIGH")
    # Phát hiện các chuyến đi bất thường
    add_flag(df_month, (df_month['trip_duration_minutes'] <= 2) & (df_month['trip_distance'] > 0), 'FLAG_DURATION_TOO_SHORT')
    add_flag(df_month, df_month['trip_duration_minutes'] > 1440, 'FLAG_DURATION_TOO_LONG') 

    # Kiểm tra khoảng cách âm hoặc khoảng cách quá xa
    add_flag(df_month, df_month['trip_distance'] <= 0, 'FLAG_DISTANCE_ZERO_OR_NEG')
    add_flag(df_month, df_month['trip_distance'] > 200, 'FLAG_DISTANCE_EXTREME') # > 200 dặm
    # Kiểm tra chuyến đi ma (không có hành khách mà vẫn tính chuyến đi)
    add_flag(df_month, (df_month["passenger_count"] <= 0) & (df_month['trip_distance'] > 0), 'FLAG_PASSENGER_INVALID')
    add_flag(df_month, df_month["fare_amount"] < 3.0, "FLAG_FARE_AMOUNT_INVALID")

    #Kiểm tra các cột có các giá trị âm (Có thể do lỗi hệ thống)
    other_fee_cols = [
    'tip_amount', 
    'tolls_amount', 
    ]


    for col in other_fee_cols:
        if col in df_month.columns:
            condition = df_month[col] < 0
            if condition.any(): 
                flag_name = f'FLAG_NEGATIVE_{col.upper()}'
                add_flag(df_month, condition, flag_name)
                print(f"Đã gắn cờ cho {condition.sum()} hàng có {col} bị âm.")


    # Check các chuyến đi bị trùng lặp dựa trên 7 khóa chính
    key_cols = ['VendorID','tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_distance', 'total_amount']
    duplicates_mask = df_month.duplicated(subset=key_cols, keep='first')
    print(f'Đã gắn cờ cho {duplicates_mask.sum()} hàng bị trùng')
    add_flag(df_month, duplicates_mask, 'FLAG_DUPLICATE')

    #
    df_month['is_clean'] = df_month['qa_flags'].apply(lambda x: len(x) == 0 or all(str(flag).startswith('FLAG_FIXED') for flag in x))
    print(f"-> Hoàn thành. {df_month['is_clean'].sum()} hàng sạch / {len(df_month)} tổng số hàng.")
    
    return df_month


In [3]:
path = r"C:\Users\APC\2526-LTXLDL-Project-1.5\raw"

## Đảm bảo thư mục tồn tại

In [ ]:
# os.makedirs('reports', exist_ok=True)
# os.makedirs('processed/clean_data_monthly', exist_ok=True) 

In [ ]:
list_of_qa_reports = [] 
for file in os.listdir(path):
    if file.endswith(".parquet"):
        filepath = os.path.join(path, file)
        print(f"\nĐang đọc tệp: {file}")
        temp_raw = pd.read_parquet(filepath)

        if "airport_fee" in temp_raw.columns:
            temp_raw = temp_raw.rename(columns={"airport_fee" : "Airport_fee"}) # Xử lý vì tháng 1 khác biệt
        
        df_full_flagged = clean_by_month(temp_raw)
        
        all_flags = df_full_flagged.explode('qa_flags')['qa_flags'].dropna()
        qa_summary_month = pd.DataFrame(all_flags.value_counts())
        qa_summary_month.columns = ['SoLuongViPham']

        month_name = file.replace('yellow_tripdata_', '').replace('.parquet', '')
        qa_summary_month['Month'] = month_name
        list_of_qa_reports.append(qa_summary_month)

        # LỌC và LƯU TỆP SẠCH cho 1 tháng 
        df_clean_month = df_full_flagged[df_full_flagged['is_clean'] == True]
        
        # Lấy cột gốc
        cols_to_drop = ['qa_flags','is_clean']
        original_cols = [col for col in df_full_flagged.columns if col not in cols_to_drop]
        df_clean_final = df_clean_month[original_cols]

        # LƯU RA TỆP .parquet CỦA THÁNG ĐÓ
        # clean_output_path = f'processed/clean_data_monthly/{month_name}_clean.parquet'
        # df_clean_final.to_parquet(clean_output_path, index=False, compression='snappy')
        # print(f"ĐÃ LƯU TỆP SẠCH: {clean_output_path}")

        # GIẢI PHÓNG BỘ NHỚ
        del temp_raw, df_full_flagged, df_clean_month, df_clean_final
        print(f"Đã giải phóng bộ nhớ của tháng {month_name}")

print("\n===================== HOÀN THÀNH ========================")


In [ ]:
if not list_of_qa_reports:
    print("Không có báo cáo QA nào để gộp.")
else:
    df_qa_annual = pd.concat(list_of_qa_reports) #Gộp

    # Xử lý lại bảng báo cáo tổng
    df_qa_annual.reset_index(inplace=True) # Chuyển index thành cột (tên cột là 'qa_flags')
    
    # Đổi tên cột 'qa_flags'
    df_qa_annual = df_qa_annual.rename(columns={'qa_flags': 'QuyTacQA'})
    #
    qa_summary_final = df_qa_annual.groupby('QuyTacQA')['SoLuongViPham'].sum().sort_values(ascending=False)
    qa_summary_final = pd.DataFrame(qa_summary_final)
    
    # Tính toán lại tỷ lệ và quyết định
    total_rows_annual = qa_summary_final['SoLuongViPham'].sum() # TỔNG SỐ LỖI
    if total_rows_annual == 0:
        qa_summary_final['TyLeViPham (%)'] = 0.0
    else:
        qa_summary_final['TyLeViPham (%)'] = (qa_summary_final['SoLuongViPham'] / total_rows_annual) * 100
    
    decisions = []
    for flag in qa_summary_final.index:
        if 'FIXED' in str(flag): decisions.append('Sửa & Giữ lại')
        else: decisions.append('Gắn cờ & Loại khỏi thống kê')
    qa_summary_final['QuyetDinhXuLy'] = decisions
    
    # Lưu tệp
    # qa_summary_path = 'reports/qa_summary_ANNUAL.csv'
    # qa_summary_final.to_csv(qa_summary_path, encoding='utf-8-sig')
    
    # print(f"ĐÃ LƯU BÁO CÁO QA TỔNG CẢ NĂM: {qa_summary_path}")
    print(qa_summary_final)
    
print("\n===================== HOÀN THÀNH ========================")

                                     SoLuongViPham  TyLeViPham (%)  \
QuyTacQA                                                             
FLAG_FIXED_TOTAL_AMOUNT                   10296991       44.944854   
FLAG_INVALID_EXTRA                         6358210       27.752653   
FLAG_FIXED_MISSINGS_RatecodeID             1309356        5.715147   
FLAG_FIXED_MISSINGS_PASSENGER              1309356        5.715147   
FLAG_FIXED_PAYMENT_TYPE                     799874        3.491333   
FLAG_DISTANCE_ZERO_OR_NEG                   773457        3.376026   
FLAG_DURATION_TOO_SHORT                     419163        1.829585   
FLAG_INVALID_IMPROVEMENT_SURCHANGE          417844        1.823828   
FLAG_FARE_AMOUNT_INVALID                    404186        1.764213   
FLAG_INVALID_MTA_TAX                        367473        1.603966   
FLAG_INVALID_CONGESTION_IMPROVEMENT         303073        1.322869   
FLAG_SPEED_TOO_HIGH                          70142        0.306160   
FLAG_INVALID_AIRPORT